# X-CLIP base/32 — DIMER E2E video-classification adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/xclip-video-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/xclip-video-classification-pipeline/blob/main/tutorials/xclip_video_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-microsoft%2Fxclip--base--patch32-ffcc4d?style=flat)](https://huggingface.co/microsoft/xclip-base-patch32) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FVideoX-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/VideoX/tree/master/X-CLIP) [![arXiv](https://img.shields.io/badge/arXiv-2208.02816-b31b1b.svg)](https://arxiv.org/abs/2208.02816)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot video classification — one 8-frame clip plus 2–32 free-text class names → a ranking of those names with a softmax over them — and bounded supervised fine-tuning of the fusion head on labelled clips of a closed label set, using the pinned `microsoft/xclip-base-patch32` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/xclip_video_classification_pipeline/`, at revision `60dfc5277df1`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `a2e27a78a2b5d802e894b8a1ef14f3a8ce490963` (~790 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `microsoft/xclip-base-patch32` snapshot (a 786 MB `model.safetensors`; no pickle is opened anywhere), fetches the first row group of the HMDB51 test shard from the Hugging Face Hub at an immutable revision with one HTTPS range request (about 113 MB; refused on any SHA-256 or byte-total mismatch), decodes the 300 clips of the ten sample classes into 8 uniformly spaced frames each with PyAV, validates the records and splits them by source video into 181 / 53 / 66, ranks five drawn clips through the inference contract with a combined input manifest and a rejection probe, scores the frozen model over the 66 held-out clips (top-1 and top-3 accuracy, macro recall and F1) beside a chance and a majority-label baseline, runs a bounded fine-tuning of the fusion head on cached tower features with validation-accuracy epoch selection, scores the held-out clips again, re-runs six held-out clips and the five drawn clips with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify ranking parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a Tesla T4 the default path took about @P:T4_TOTAL_MIN@ minutes of cell time (eight epochs @P:T4_ADAPT_S@ s, frozen scoring of 66 clips @P:T4_FROZEN_S@ s); a CUDA runtime is used automatically when present, and the path is practical on CPU too (the build venv decoded the 300 clips in about 70 s, scored the test split in 10 s and ran the eight epochs in about 60 s).

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip of clips (AVI, MP4, MOV, MKV, WebM, GIF or WebP; at least 8 frames each) plus a `labels.csv` (`file`, `label`, optional `id` and `source`; one row per clip, at least sixteen clips of at least two labels with at least two clips each). The records pass through the same validation, source-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the HMDB51 sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

X-CLIP base/32 is a video–text model: a CLIP ViT-B/32 frame encoder with cross-frame attention embeds each of the 8 frames of a clip, a one-layer multi-frame integration transformer fuses the 8 frame embeddings into one video embedding, a CLIP text encoder embeds each class name, and a video-specific prompt generator conditions the text embeddings on the clip's patch features; the clip is scored against every name and the carried module softmaxes the scores over the names you supplied (196,585,729 parameters in all, trained fully supervised on Kinetics-400, published under the **MIT** licence). **The probabilities are a softmax over your label set**: a relative ranking that sums to 1, not a calibrated probability, and a set that omits the true class still yields a confident top-1.

What this notebook adds to inference is **adaptation of a closed label set on labelled clips**. The clips are 300 human-action clips of ten HMDB51 classes — brushing hair, doing a cartwheel, catching a ball, chewing, clapping hands, climbing, climbing stairs, diving into water, drawing a sword, dribbling a basketball — cut from films and web videos; the frozen model already ranks the right class first for **@P:FROZEN_TOP1@** of the 66 held-out clips in the build record (chance is 0.100), so the honest question is narrow: does a bounded fine-tuning of the fusion head — the frame-integration transformer, the two visual projections and the prompt generator, 10,247,680 of the parameters — on 181 clips move the held-out **top-1 accuracy**, **top-3 accuracy**, **macro recall** and **macro F1** on a source-disjoint test split past the frozen model and two **non-adapted baselines**, and what does it do to the drawn clips the same head ranks? Nothing here is a claim about your videos or your classes: it is one seeded split of one small labelled set.

**Snapshot note:** the pinned revision ships `model.safetensors` (a 9-file manifest with the tokenizer and processor files) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the processor or the model is constructed. The pipeline runs in **float32 on every device**: the adapter is trained in float32 and overlays without a cast, and CPU, Tesla-class and consumer GPUs then run the same arithmetic.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled clip set, decode it into 8-frame clips, validate it and split it by source video without leakage; rank drawn clips through the public API and read the output contract correctly (a softmax over the supplied names, no calibrated probability, a `sample-sanity` report only when the true classes are known); measure the frozen model's held-out top-1 / top-3 accuracy, macro recall and F1 beside two non-adapted baselines; run a bounded fine-tuning of the fusion head with the cross-entropy over the closed label set, explicit hyperparameters and validation-based epoch selection; evaluate on a source-disjoint test split; look at the adapted rankings next to the frozen ones and the references, and at what the drawn clips do after the shared head was tuned; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** temporal localisation or per-frame labels (one ranking per clip), open-set or abstaining classification (the softmax always picks one of your names), video–text retrieval over a corpus, fine-tuning of the vision tower, the text tower, the text projection or the logit scale, evaluation on Kinetics-400 or the full HMDB51 protocol (only one seeded 300-clip sample of ten classes is scored here), non-English class names, and any claim that ten HMDB51 actions stand in for your videos. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Kaggle, Python 3.12; CPU or CUDA). The default path uses CUDA automatically when present. The vision tower runs `EVAL_BATCH_SIZE` clips (8 × 8 frames) per forward and the build record measured @P:T4_FROZEN_S@ s to score 66 clips and @P:T4_ADAPT_S@ s for the eight epochs (caching the tower features for 181 + 53 clips took @P:T4_CACHE_S@ s) on a Tesla T4, about @P:T4_TOTAL_MIN@ minutes of cell time for the whole path including the pinned install and the downloads; the build venv's CPU ran the same path in a few minutes. The pinned `torch==2.14.0` install, the 786 MB checkpoint and the 113 MB row group are the large downloads of the run.
- **Knowledge:** basic Python, NumPy and PIL; what a contrastive video–text model scores and why a softmax over a label set you chose is a ranking and not a probability; what top-1 accuracy, macro recall and macro F1 measure on a closed label set and why 66 clips give no dispersion; why clips cut from one source video must stay in one split.
- **Data contract:** records are `{id, frames, label}` — `frames` exactly `NUM_FRAMES` (8) PIL frames of one size with sides within 16..4,096 px (a longer clip is subsampled uniformly by `sample_frames`; a container file is decoded by `decode_clip`), `label` one of the closed label set (normalised like the candidate names: stripped, lower-cased, trailing full stop removed), and an optional `source` naming the video the clip was cut from. Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 16..5,000 records, at least two labels and at least two clips per label; splitting de-duplicates by decoded pixels and keeps every clip of one source in one split. BYOD accepts one zip (or directory) of clips plus a `labels.csv` in the layout named above.
- **Validation is structural, not semantic:** every clip is decoded and every label checked against the set, but nothing checks that a label describes its clip — a mislabelled set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads row group 0 of `default/test/0000.parquet` from `https://huggingface.co/datasets/mteb/HMDB51/resolve/<revision>/` at the immutable parquet-conversion revision `50bb2abb…` with HTTPS range requests (the parquet footer plus about 113 MB of row-group bytes out of a 481 MB shard), pinned by SHA-256 and byte total in the carried `samples.py` and refused on any mismatch. HMDB51 is published under the CC BY 4.0 licence (Serre Lab, Brown University; Kuehne et al. 2011); the `mteb/HMDB51` repository is a parquet repack of it; nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `microsoft/xclip-base-patch32` snapshot (~790 MB in total) at revision `a2e27a78a2b5…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `av` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
    'av==18.1.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'xclip-video-classification-pipeline',
    'repository_revision': '60dfc5277df17826d208500033bde5a38c055b69',
    'embedded_module': 'src/xclip_video_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/xclip_video_classification_pipeline/pipeline.py', 'src/xclip_video_classification_pipeline/metrics.py', 'src/xclip_video_classification_pipeline/samples.py'],
    'module_sha256': '2d7d1ab29e34a29a83cfc9c4d2499998606c6a4148649d456ec30b8b44968547',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, av
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'av': av.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/xclip_video_classification_pipeline/` @ `60dfc5277df1`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/xclip_video_classification_pipeline/pipeline.py`

In [ ]:
"""Zero-shot video classification with the pinned ``microsoft/xclip-base-patch32`` checkpoint (X-CLIP), plus the
adaptation contract for a closed label set: corpus evaluation of labelled clips, bounded fine-tuning of the fusion
head (the frame-integration transformer, the two visual projections and the video-specific prompt generator) on
cached tower features, and a verified adapter artifact.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the X-CLIP architecture comes from the pinned ``transformers`` release, the
weights are SafeTensors, and no model-repository code is executed. A clip is a sequence of exactly
NUM_FRAMES PIL frames; the caller names the candidate classes as free text and receives a softmax over
those names — a relative ranking, not a calibrated probability.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

import hashlib
import json
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image, ImageSequence

MODEL_ID = "microsoft/xclip-base-patch32"
MODEL_REVISION = "a2e27a78a2b5d802e894b8a1ef14f3a8ce490963"
MODEL_LICENSE = "mit"
MODEL_KEY = "xclip-base-patch32"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The checkpoint was trained on 8 frames per clip (config.json vision_config.num_frames); the temporal
# modules expect exactly that many, so a clip is exactly NUM_FRAMES frames and longer sequences are
# subsampled uniformly by sample_frames.
NUM_FRAMES = 8
# Frame preprocessing: shorter side resized to 224, centre crop 224x224, ImageNet mean/std
# (preprocessor_config.json), 32x32 patches -> 49 tokens per frame.
FRAME_SIZE = 224
# Input ceilings. Each label is one CLIP text query (77-token context); the label set is the caller's
# closed vocabulary for this request.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MIN_LABELS = 2
MAX_LABELS = 32
MAX_LABEL_CHARS = 64
MAX_TEXT_TOKENS = 77
WEIGHTS_FILE = "model.safetensors"

# Adaptation contract: the fusion head — the multi-frame integration transformer (`mit`), the two visual projections
# and the video-specific prompt generator — is the adapter; the vision tower (12 ViT-B/32 layers with cross-frame
# attention), the text tower, the text projection and the logit scale stay frozen, so their outputs are computed
# once per clip and label set and cached.
PARAMETER_COUNT = 196_585_729
HEAD_PARAMETERS = 10_247_680
FRAME_TOKENS = 50  # 1 CLS + 49 patch tokens per 224-px frame
VISION_WIDTH = 768
_TRAINABLE_PREFIXES = ("visual_projection.", "mit.", "prompts_visual_layernorm.", "prompts_visual_projection", "prompts_generator.")
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = 1
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
MIN_SCORED_RECORDS = 30  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 5_000
EVAL_BATCH_SIZE = 8  # clips per vision-tower forward (8 clips x NUM_FRAMES frames)
GRAD_CLIP = 1.0


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _weight_digest(root: Path) -> str | None:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        return None
    with open(manifest_path, encoding="utf-8") as handle:
        entries = json.load(handle).get("files", [])
    return next((e["sha256"] for e in entries if e["path"] == WEIGHTS_FILE), None)


def _trainable_names(model: Any) -> list[str]:
    """The fusion head's tensors; the vision tower, the text tower, the text projection and the logit scale stay
    frozen."""
    return [name for name, _ in model.named_parameters() if name.startswith(_TRAINABLE_PREFIXES)]


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}"
        )
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith(_TRAINABLE_PREFIXES) for n in names):
        raise ValueError("artifact tensors must all belong to the fusion head (mit, visual projections, prompt generator)")
    adapter = manifest.get("adapter") or {}
    labels = adapter.get("labels")
    if not isinstance(labels, list) or len(labels) < MIN_LABELS:
        raise ValueError("artifact manifest must record adapter.labels, the closed label set the head was trained on")


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def format_labels(labels: Sequence[str]) -> list[str]:
    """Validate the candidate class names and normalise them: stripped, whitespace-collapsed,
    lower-cased, trailing full stop removed, distinct. Names are passed to the CLIP text tower as-is
    otherwise (the upstream zero-shot recipe uses plain class names such as "playing soccer")."""
    if isinstance(labels, str) or not isinstance(labels, Sequence):
        raise TypeError("labels must be a list of class names, not a single string")
    if not MIN_LABELS <= len(labels) <= MAX_LABELS:
        raise ValueError(
            f"label count {len(labels)} outside MIN_LABELS {MIN_LABELS}..MAX_LABELS {MAX_LABELS}"
        )
    cleaned: list[str] = []
    for name in labels:
        if not isinstance(name, str):
            raise TypeError(f"label must be str, got {type(name).__name__}")
        text = " ".join(name.split()).strip().rstrip(".").strip().lower()
        if not text:
            raise ValueError("labels must not be empty")
        if len(text) > MAX_LABEL_CHARS:
            raise ValueError(
                f"label {text[:12]!r}... is {len(text)} chars > MAX_LABEL_CHARS {MAX_LABEL_CHARS}"
            )
        cleaned.append(text)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("labels must be distinct after normalisation")
    return cleaned


def validate_frame(frame: Any) -> Image.Image:
    if not isinstance(frame, Image.Image):
        raise TypeError(f"frame must be a PIL.Image.Image, got {type(frame).__name__}")
    width, height = frame.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"frame side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"frame side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return frame.convert("RGB")


def validate_clip(frames: Any) -> list[Image.Image]:
    """Exactly NUM_FRAMES PIL frames of one common size, each within the side ceilings."""
    if isinstance(frames, Image.Image) or not isinstance(frames, Sequence):
        raise TypeError("frames must be a sequence of PIL.Image.Image, not a single image")
    if len(frames) != NUM_FRAMES:
        raise ValueError(
            f"a clip is exactly NUM_FRAMES={NUM_FRAMES} frames, got {len(frames)}; "
            "use sample_frames to subsample a longer sequence"
        )
    rgb = [validate_frame(frame) for frame in frames]
    if len({frame.size for frame in rgb}) != 1:
        raise ValueError("all frames of a clip must have the same size")
    return rgb


def sample_frames(frames: Sequence[Image.Image], n: int = NUM_FRAMES) -> list[Image.Image]:
    """Pick ``n`` frames at evenly spaced indices (first and last included) from a longer sequence."""
    if isinstance(frames, Image.Image) or not isinstance(frames, Sequence):
        raise TypeError("frames must be a sequence of PIL.Image.Image")
    if len(frames) < n:
        raise ValueError(f"need at least {n} frames to sample {n}, got {len(frames)}")
    indices = np.linspace(0, len(frames) - 1, num=n).round().astype(int)
    return [frames[int(index)] for index in indices]


def frames_from_animation(image: Image.Image) -> list[Image.Image]:
    """Decode every frame of an animated image (GIF, WebP, APNG) that Pillow can open into RGB copies."""
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    frames = [frame.convert("RGB") for frame in ImageSequence.Iterator(image)]
    if not frames:
        raise ValueError("the image holds no frames")
    return frames


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        f"one clip of exactly NUM_FRAMES={NUM_FRAMES} PIL.Image.Image frames of one size (any mode, "
        "converted to RGB) plus MIN_LABELS..MAX_LABELS free-text class names"
    ),
    "frame_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "frames_per_clip": NUM_FRAMES,
    "labels": [MIN_LABELS, MAX_LABELS],
    "label_chars": [1, MAX_LABEL_CHARS],
    "label_tokens": [1, MAX_TEXT_TOKENS],
    "preprocessing": (
        f"each frame: shorter side resized to {FRAME_SIZE}, centre crop {FRAME_SIZE}x{FRAME_SIZE}, ImageNet "
        "mean/std, 32x32 patches; labels normalised into one CLIP text query each (format_labels); the "
        "video embedding (frame features fused by the multi-frame integration transformer) is scored "
        "against each label embedding and the scores are softmaxed over the supplied labels"
    ),
    "output": (
        "one probability per supplied label (softmax over the label set: a relative ranking that sums to "
        "1 and is not calibrated), the raw logits, and the top-1 label"
    ),
}


def _check_inputs(frames: Any, labels: Any) -> tuple[list[Image.Image], list[str]]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``classify`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    return validate_clip(frames), format_labels(labels)


def validate_inputs(
    clips: Sequence[Sequence[Image.Image]],
    labels: Sequence[str],
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every clip is checked exactly as ``classify`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(clips, Image.Image) or not isinstance(clips, Sequence) or not clips:
        raise TypeError("clips must be a non-empty sequence of frame sequences")
    if clips and isinstance(clips[0], Image.Image):
        raise TypeError("clips must be a sequence of clips (each a sequence of frames), not one clip")
    if names is not None and len(names) != len(clips):
        raise ValueError(f"names has {len(names)} entries for {len(clips)} clips")
    checked_labels = format_labels(labels)
    observed = []
    for index, clip in enumerate(clips):
        rgb, _ = _check_inputs(clip, labels)
        observed.append(
            {
                "id": names[index] if names else f"clip-{index}",
                "n_frames": len(rgb),
                "frame_mode": clip[0].mode,
                "frame_size": list(rgb[0].size),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": observed,
        "labels": checked_labels,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    correct_labels: Sequence[str] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``correct_labels`` (one class name per result, in order; normalised like the labels) the report
    carries ``top1_accuracy`` over the clips, the chance baseline (mean of 1/n_labels) and one
    per-clip entry, verdict ``sample-sanity``; without them it is ``not-measurable`` and says what
    labelled data would make the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one classification result")
    base = {
        "task": "zero-shot video classification over a caller-supplied label set (top-1 over the labels)",
        "score_semantics": (
            "probabilities are a softmax over the supplied labels only: a relative ranking that sums to 1, "
            "not a calibrated probability, and a label set without the true class still yields a confident "
            "top-1"
        ),
        "sample_kind": sample_kind,
        "n_clips": len(results),
        "n_labels": [len(result["labels"]) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if correct_labels is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no correct labels were supplied for the classified clips",
            "needs": (
                "labelled clips from the deployment domain with a class vocabulary matching the labels "
                "(Kinetics-style annotations) scored with top-1/top-5 accuracy over many clips; no such "
                "labelled set ships with this repository"
            ),
        }
    if len(correct_labels) != len(results):
        raise ValueError(f"correct_labels has {len(correct_labels)} entries for {len(results)} results")
    per_clip = []
    for result, correct in zip(results, correct_labels, strict=True):
        key = format_labels([correct, "__second__"])[0]
        if key not in result["labels"]:
            raise ValueError(f"correct label {correct!r} is not among the result's labels")
        per_clip.append(
            {
                "clip": result.get("clip"),
                "top1": result["top1"],
                "top1_probability": result["predictions"][0]["probability"],
                "correct_label": key,
                "correct": result["top1"] == key,
                "rank_of_correct": next(
                    index for index, entry in enumerate(result["predictions"]) if entry["label"] == key
                )
                + 1,
            }
        )
    chance = sum(1.0 / n for n in base["n_labels"]) / len(results)
    return {
        **base,
        "metrics": [
            {
                "id": "top1_accuracy",
                "value": sum(entry["correct"] for entry in per_clip) / len(per_clip),
                "estimation": f"{len(per_clip)} clip(s), no dispersion estimate",
            }
        ],
        "baselines": [{"id": "chance", "value": chance, "note": "mean of 1/n_labels over the clips"}],
        "per_clip": per_clip,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_clip)} clip(s) with caller-known classes; plumbing evidence, not a video "
            "classification benchmark"
        ),
        "needs": (
            "labelled clips from the deployment domain with a matching class vocabulary for any "
            "top-1/top-5 accuracy claim; Kinetics-400 and UCF101 are not bundled"
        ),
    }


@dataclass
class XClipVideoClassificationPipeline:
    """Zero-shot video classification over the pinned X-CLIP base/32 checkpoint."""

    _runner: Callable[[list[Image.Image], list[str]], np.ndarray]
    device: str
    _batch_runner: Callable[[list[list[Image.Image]], list[str]], np.ndarray] | None = None
    _model: Any = None
    _processor: Any = None
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> XClipVideoClassificationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import XCLIPModel, XCLIPProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        weight_sha256 = _weight_digest(root) if (root / MANIFEST_NAME).is_file() else None
        processor = XCLIPProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = XCLIPModel.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float32, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(frames: list[Image.Image], labels: list[str]) -> np.ndarray:
            # The frame list is handed to the snapshot's VideoMAE-style image processor directly: in
            # the pinned transformers release XCLIPProcessor's `videos=` keyword yields no
            # pixel_values (only `images=` does), and calling the two sub-processors makes the
            # contract explicit. One clip -> pixel_values (1, NUM_FRAMES, 3, 224, 224).
            pixel_values = processor.image_processor([frames], return_tensors="pt")["pixel_values"]
            text = processor.tokenizer(labels, padding=True, return_tensors="pt")
            with torch.inference_mode():
                outputs = model(
                    pixel_values=pixel_values.to(resolved_device),
                    input_ids=text["input_ids"].to(resolved_device),
                    attention_mask=text["attention_mask"].to(resolved_device),
                )
            return outputs.logits_per_video[0].float().cpu().numpy()

        def batch_runner(clips: list[list[Image.Image]], labels: list[str]) -> np.ndarray:
            pixel_values = processor.image_processor(clips, return_tensors="pt")["pixel_values"]
            text = processor.tokenizer(labels, padding=True, return_tensors="pt")
            with torch.inference_mode():
                outputs = model(
                    pixel_values=pixel_values.to(resolved_device),
                    input_ids=text["input_ids"].to(resolved_device),
                    attention_mask=text["attention_mask"].to(resolved_device),
                )
            return outputs.logits_per_video.float().cpu().numpy()

        return cls(runner, resolved_device, batch_runner, model, processor, weight_sha256, None)

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained")
        return self._model, self._processor

    def classify(self, frames: Sequence[Image.Image], labels: Sequence[str]) -> dict[str, Any]:
        """Rank ``labels`` for one clip of exactly NUM_FRAMES frames; probabilities are a softmax over them.

        The softmax is a relative ranking over the supplied labels, not a calibrated probability.
        """
        rgb, names = _check_inputs(frames, labels)
        logits = np.asarray(self._runner(rgb, names), dtype=np.float64).reshape(-1)
        if logits.shape != (len(names),) or not np.all(np.isfinite(logits)):
            raise RuntimeError(f"backend returned logits of shape {logits.shape} for {len(names)} labels")
        shifted = np.exp(logits - logits.max())
        probs = shifted / shifted.sum()
        order = np.argsort(-probs)
        predictions = [
            {"label": names[index], "probability": float(probs[index]), "logit": float(logits[index])}
            for index in order
        ]
        return {
            "predictions": predictions,
            "top1": predictions[0]["label"],
            "labels": names,
            "n_frames": len(rgb),
            "frame_size": list(rgb[0].size),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ------------------------------------------------------------------------------------------------------
    # Adaptation contract: batched classification, corpus evaluation, bounded fine-tuning, artifacts
    # ------------------------------------------------------------------------------------------------------

    @staticmethod
    def _result(logits: np.ndarray, names: list[str]) -> dict[str, Any]:
        logits = np.asarray(logits, dtype=np.float64).reshape(-1)
        if logits.shape != (len(names),) or not np.all(np.isfinite(logits)):
            raise RuntimeError(f"backend returned logits of shape {logits.shape} for {len(names)} labels")
        shifted = np.exp(logits - logits.max())
        probs = shifted / shifted.sum()
        order = np.argsort(-probs, kind="stable")
        predictions = [
            {"label": names[index], "probability": float(probs[index]), "logit": float(logits[index])}
            for index in order
        ]
        return {"predictions": predictions, "top1": predictions[0]["label"], "labels": names}

    def classify_batch(
        self,
        clips: Sequence[Sequence[Image.Image]],
        labels: Sequence[str],
        *,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> list[dict[str, Any]]:
        """Rank `labels` for many clips, `batch_size` clips per forward; one result (as `classify` returns it, minus
        the frame fields) per clip, in order. With an injected runner and no batch runner the clips are ranked one by
        one through the runner."""
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        names = format_labels(labels)
        checked = [validate_clip(clip) for clip in clips]
        out: list[dict[str, Any]] = []
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            if self._batch_runner is not None:
                logits = np.asarray(self._batch_runner(batch, names), dtype=np.float64)
                if logits.shape != (len(batch), len(names)):
                    raise RuntimeError(f"backend returned logits of shape {logits.shape} for {len(batch)} clips x {len(names)} labels")
                rows = list(logits)
            else:
                rows = [np.asarray(self._runner(clip, names), dtype=np.float64) for clip in batch]
            for clip, row in zip(batch, rows, strict=True):
                out.append({**self._result(row, names), "n_frames": len(clip), "frame_size": list(clip[0].size)})
            if progress is not None:
                progress(min(start + batch_size, len(checked)), len(checked))
        return out

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        labels: Sequence[str],
        *,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> dict[str, Any]:
        """Rank the closed label set for every validated record and score the rankings against the record labels
        with ``metrics.classification_metrics`` (top-1 / top-3 accuracy, macro recall and F1, confusion). Works with
        an injected runner too."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, labels, min_records=1, max_records=MAX_EVAL_RECORDS, min_per_class=1)["records"]
        names = format_labels(labels)
        started = time.perf_counter()
        results = self.classify_batch([r["frames"] for r in checked], names, batch_size=batch_size, progress=progress)
        rankings = [[p["label"] for p in r["predictions"]] for r in results]
        metrics = classification_metrics(rankings, checked, names)
        return {
            **metrics,
            "labels": names,
            "predictions": [r["top1"] for r in results],
            "top1_probability": [r["predictions"][0]["probability"] for r in results],
            "rankings": rankings,
            "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
            "adapted": self.adapter is not None,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def _encode_clips(self, records: Sequence[Mapping[str, Any]], batch_size: int, progress: Callable[[int, int], None] | None = None) -> tuple[Any, Any]:
        """Run the frozen vision tower once per clip: pooled CLS features (n, NUM_FRAMES, VISION_WIDTH) and patch
        features (n, NUM_FRAMES, FRAME_TOKENS - 1, VISION_WIDTH), kept on the model's device in float32."""
        model, processor = self._require_model()
        import torch

        cls_out, patch_out = [], []
        with torch.no_grad():  # not inference_mode: the cached tensors feed autograd during adapt
            for start in range(0, len(records), batch_size):
                batch = [r["frames"] for r in records[start : start + batch_size]]
                pixel_values = processor.image_processor(batch, return_tensors="pt")["pixel_values"].to(self.device)
                vision = model.vision_model(pixel_values=pixel_values.flatten(0, 1))
                cls_out.append(vision[1].reshape(len(batch), NUM_FRAMES, -1).clone())
                patch_out.append(vision[0][:, 1:, :].reshape(len(batch), NUM_FRAMES, FRAME_TOKENS - 1, -1).clone())
                if progress is not None:
                    progress(min(start + batch_size, len(records)), len(records))
        return torch.cat(cls_out), torch.cat(patch_out)

    def _encode_labels(self, names: Sequence[str]) -> Any:
        model, processor = self._require_model()
        import torch

        text = processor.tokenizer(list(names), padding=True, return_tensors="pt")
        with torch.no_grad():
            return model.get_text_features(input_ids=text["input_ids"].to(self.device), attention_mask=text["attention_mask"].to(self.device)).clone()

    def _head_logits(self, cls_features: Any, patch_features: Any, text_features: Any) -> Any:
        """The fusion head on cached tower features: exactly what `XCLIPModel.forward` computes after the towers
        (parity with the full forward is asserted by the model-backed tests)."""
        model, _ = self._require_model()
        import torch

        batch = cls_features.shape[0]
        video = model.visual_projection(cls_features.reshape(batch * NUM_FRAMES, -1)).view(batch, NUM_FRAMES, -1)
        video = model.mit(video)[1]
        img = model.prompts_visual_layernorm(patch_features.reshape(batch * NUM_FRAMES, FRAME_TOKENS - 1, -1))
        img = (img @ model.prompts_visual_projection).view(batch, NUM_FRAMES, -1, video.shape[-1]).mean(dim=1)
        text = text_features.unsqueeze(0).expand(batch, -1, -1)
        text = text + model.prompts_generator(text, img)
        video = video / video.norm(p=2, dim=-1, keepdim=True)
        text = text / text.norm(p=2, dim=-1, keepdim=True)
        return torch.einsum("bd,bkd->bk", video, text) * model.logit_scale.exp()

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None,
        labels: Sequence[str],
        *,
        epochs: int = 8,
        lr: float = 1e-5,
        batch_size: int = 16,
        seed: int = 0,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the fusion head on labelled clips with the cross-entropy over the closed label set
        (the model's own contrastive scoring, read as a classifier over `labels`). The frozen towers — the ViT-B/32
        vision encoder with its cross-frame attention and the CLIP text encoder — are run once per clip and label
        set under no gradient and their outputs are cached, so each step runs only the head; the logits equal the
        full model's exactly. AdamW (no weight decay), gradient clipping at `GRAD_CLIP`, seeded shuffling, no
        scheduler, no augmentation. Epoch 0 records the frozen model's validation metrics; the epoch with the highest
        validation top-1 accuracy (the earliest on ties) is kept. On any exception the frozen head is restored."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
            raise ValueError("epochs must be an int in 1..100")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 128:
            raise ValueError("batch_size must be an int in 1..128")
        if not isinstance(lr, int | float) or isinstance(lr, bool) or not 0 < lr <= 1e-2:
            raise ValueError("lr must be a number in (0, 1e-2]")
        names = format_labels(labels)
        train_checked = validate_dataset(train, names)["records"]
        val_checked = validate_dataset(val, names, min_records=1, min_per_class=1)["records"] if val is not None else None
        model, _processor = self._require_model()
        import torch

        started = time.perf_counter()
        head_names = _trainable_names(model)
        params = {name: param for name, param in model.named_parameters() if name in set(head_names)}
        n_trainable = sum(p.numel() for p in params.values())
        for name, param in model.named_parameters():
            param.requires_grad_(name in params)
        backup = {name: param.detach().clone() for name, param in params.items()}
        cudnn_flags = torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
        try:
            model.eval()
            text_features = self._encode_labels(names)
            targets = torch.tensor([names.index(r["label"]) for r in train_checked], device=self.device)
            cls_train, patch_train = self._encode_clips(train_checked, EVAL_BATCH_SIZE)
            cached = None
            if val_checked is not None:
                cached = self._encode_clips(val_checked, EVAL_BATCH_SIZE)
            cache_seconds = round(time.perf_counter() - started, 3)

            def score_val() -> dict[str, Any] | None:
                if val_checked is None or cached is None:
                    return None
                with torch.no_grad():
                    rows = []
                    for start in range(0, len(val_checked), EVAL_BATCH_SIZE):
                        rows.append(self._head_logits(cached[0][start : start + EVAL_BATCH_SIZE], cached[1][start : start + EVAL_BATCH_SIZE], text_features))
                    logits = torch.cat(rows).float().cpu().numpy()
                rankings = [[names[i] for i in np.argsort(-row, kind="stable")] for row in logits]
                m = classification_metrics(rankings, val_checked, names)
                return {k: m[k] for k in ("top1_accuracy", "top3_accuracy", "macro_recall", "macro_f1", "n")}

            history: list[dict[str, Any]] = [{"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}]
            if progress is not None:
                progress(history[-1])
            best_epoch, best_acc = 0, (history[0]["val"] or {}).get("top1_accuracy", -1.0)
            best_state = {name: param.detach().clone() for name, param in params.items()}
            optimizer = torch.optim.AdamW(list(params.values()), lr=lr, weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            order = list(range(len(train_checked)))
            for epoch in range(1, epochs + 1):
                rng.shuffle(order)
                model.train()
                total, steps = 0.0, 0
                for start in range(0, len(order), batch_size):
                    idx = torch.tensor(order[start : start + batch_size], device=self.device)
                    optimizer.zero_grad(set_to_none=True)
                    logits = self._head_logits(cls_train[idx], patch_train[idx], text_features)
                    loss = torch.nn.functional.cross_entropy(logits, targets[idx])
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(list(params.values()), GRAD_CLIP)
                    optimizer.step()
                    total += float(loss.detach())
                    steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": round(total / max(steps, 1), 5), "val": score_val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                acc = (entry["val"] or {}).get("top1_accuracy")
                if val_checked is None or (acc is not None and acc > best_acc):
                    best_epoch, best_acc = epoch, acc if acc is not None else best_acc
                    best_state = {name: param.detach().clone() for name, param in params.items()}
            with torch.no_grad():
                for name, param in params.items():
                    param.copy_(best_state[name])
        except BaseException:
            with torch.no_grad():
                for name, param in params.items():
                    param.copy_(backup[name])
            model.eval()
            raise
        finally:
            for param in model.parameters():
                param.requires_grad_(False)
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags
        self.adapter = {
            "labels": names,
            "trainable_names": head_names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "batch_size": batch_size,
            "best_epoch": best_epoch,
            "selection": "highest validation top-1 accuracy" if val_checked is not None else "final epoch (no validation split)",
            "lr": lr,
            "seed": seed,
            "grad_clip": GRAD_CLIP,
            "objective": "cross-entropy over the closed label set on cached tower features",
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "cache_seconds": cache_seconds,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained tensors as safetensors plus a manifest naming the base, the digests, the label set and
        the training configuration. Requires a prior `adapt`."""
        model, _processor = self._require_model()  # refuse before importing torch
        import torch
        from safetensors.torch import save_file

        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHTS_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the fusion head."""
        model, _processor = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model)
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> XClipVideoClassificationPipeline:
        """Check the adapter manifest against the base snapshot's recorded weight digest, load the verified base, then
        overlay the adapter (checked again, and the tensor set, before deserialising). A refused manifest never loads
        a model."""
        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        _check_artifact_manifest(manifest, artifact, _weight_digest(root) or "")
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/xclip_video_classification_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level measures for closed-set clip classification: top-1 and top-3 accuracy, macro recall and macro F1
over the label set, the per-label confusion, and two non-adapted baselines (chance, majority-training-label).

Every rate is computed over the supplied records with the supplied closed label set; nothing is calibrated and no
dispersion is estimated (one seeded split of one sample gives one number)."""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

from collections import Counter
from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import format_labels` removed — names are kernel globals defined by the carried modules

TOP_K = 3

METRIC_DEFINITIONS = {
    "top1_accuracy": "fraction of clips whose highest-scoring label is the reference label",
    "top3_accuracy": f"fraction of clips whose reference label is among the {TOP_K} highest-scoring labels (1.0 by construction when the label set has {TOP_K} or fewer labels)",
    "macro_recall": "mean over the label set of the per-label recall (correct clips of that label / clips of that label); labels absent from the records are skipped",
    "macro_f1": "mean over the label set of the per-label F1 between precision and recall of the top-1 decision; labels absent from both references and predictions are skipped",
    "confusion": "reference label -> predicted top-1 label -> count",
    "baselines": "chance = 1 / |labels| top-1 by construction; majority = the most frequent training label predicted for every clip",
}


def _rank(ranking: Sequence[str], reference: str) -> int:
    for index, label in enumerate(ranking):
        if label == reference:
            return index + 1
    return len(ranking) + 1


def classification_metrics(
    rankings: Sequence[Sequence[str]], records: Sequence[Mapping[str, Any]], labels: Sequence[str]
) -> dict[str, Any]:
    """Score one ranking of `labels` (best first) per record against the record's `label`."""
    label_set = list(format_labels(labels))
    if len(rankings) != len(records):
        raise ValueError(f"{len(rankings)} rankings for {len(records)} records")
    if not records:
        raise ValueError("no records to score")
    refs = [format_labels([r["label"], "__other__"])[0] for r in records]
    for ref in refs:
        if ref not in label_set:
            raise ValueError(f"record label {ref!r} is not in the label set")
    ranks = [_rank(list(ranking), ref) for ranking, ref in zip(rankings, refs, strict=True)]
    tops = [list(ranking)[0] if ranking else "" for ranking in rankings]
    confusion: dict[str, dict[str, int]] = {label: {} for label in label_set}
    for ref, top in zip(refs, tops, strict=True):
        confusion[ref][top] = confusion[ref].get(top, 0) + 1
    support = Counter(refs)
    predicted = Counter(tops)
    recalls, f1s = [], []
    per_label = {}
    for label in label_set:
        tp = confusion[label].get(label, 0)
        n_ref, n_pred = support.get(label, 0), predicted.get(label, 0)
        recall = tp / n_ref if n_ref else None
        precision = tp / n_pred if n_pred else None
        f1 = None
        if n_ref or n_pred:
            f1 = (2 * tp / (n_ref + n_pred)) if (n_ref + n_pred) else None
        per_label[label] = {"support": n_ref, "predicted": n_pred, "correct": tp, "recall": recall, "precision": precision, "f1": f1}
        if recall is not None:
            recalls.append(recall)
        if f1 is not None:
            f1s.append(f1)
    k = min(TOP_K, len(label_set))
    return {
        "n": len(records),
        "n_labels": len(label_set),
        "top1_accuracy": sum(r == 1 for r in ranks) / len(ranks),
        "top3_accuracy": sum(r <= k for r in ranks) / len(ranks),
        "macro_recall": sum(recalls) / len(recalls) if recalls else 0.0,
        "macro_f1": sum(f1s) / len(f1s) if f1s else 0.0,
        "mean_rank": sum(ranks) / len(ranks),
        "per_label": per_label,
        "confusion": confusion,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def chance_baseline(records: Sequence[Mapping[str, Any]], labels: Sequence[str]) -> dict[str, Any]:
    """Uniform guessing over the label set: top-1 = 1 / |labels|, top-3 = min(3, |labels|) / |labels|."""
    label_set = list(format_labels(labels))
    n = len(label_set)
    return {
        "n": len(records),
        "n_labels": n,
        "top1_accuracy": 1.0 / n,
        "top3_accuracy": min(TOP_K, n) / n,
        "macro_recall": 1.0 / n,
        "macro_f1": 1.0 / n,
        "kind": "chance",
    }


def majority_label(train: Sequence[Mapping[str, Any]], labels: Sequence[str]) -> str:
    """The most frequent training label (ties broken by label-set order)."""
    label_set = list(format_labels(labels))
    counts = Counter(format_labels([r["label"], "__other__"])[0] for r in train)
    return max(label_set, key=lambda label: (counts.get(label, 0), -label_set.index(label)))


def majority_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]], labels: Sequence[str]
) -> dict[str, Any]:
    """Predict the most frequent training label for every clip, ranking the rest in label-set order."""
    label_set = list(format_labels(labels))
    top = majority_label(train, label_set)
    ranking = [top] + [label for label in label_set if label != top]
    out = classification_metrics([ranking] * len(records), records, label_set)
    out["kind"] = "majority"
    out["label"] = top
    return out

**Module 3/3:** `src/xclip_video_classification_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled clip datasets for the adaptation contract: the digest-pinned HMDB51 sample, the record contract and its
structural validation, source-disjoint splitting, and the BYOD loader.

A record is ``{id, frames, label}`` where ``frames`` is a list of exactly ``NUM_FRAMES`` PIL frames of one size
(uniformly sampled from a clip; sides within the pipeline's ceilings) and ``label`` the class name the clip is asked
to be ranked under — one of the caller's closed label set, normalised like ``format_labels`` normalises the
candidate names. An optional ``source`` names the video the clip was cut from; splits are made disjoint on it.

The default sample is drawn from HMDB51 (Kuehne et al. 2011; Serre Lab, Brown University; **CC BY 4.0**) as
repacked into parquet on the Hugging Face Hub (``mteb/HMDB51``) at an immutable revision: the first row group of
the test shard is read with one HTTPS range request (about 113 MB; the shard's declared size is checked first and
the row group's content is refused unless its SHA-256 matches the pin). It holds every test clip of the first ten
action classes (30 each, 300 clips) plus 18 clips of the eleventh, which are dropped so every class is complete.
Clips are MPEG-4 AVI files of 76..647 frames at 30 fps, mostly 320×240, decoded with PyAV into ``NUM_FRAMES``
uniformly spaced frames. Several clips are cut from one source video (its name is the prefix of the file name), so
the sample is split by source, never by clip.
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import random
import re
import urllib.request
import zipfile
from collections import defaultdict
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MODEL_ID, NUM_FRAMES, format_labels, sample_frames, validate_clip` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "HMDB51 (test shard), first parquet row group: the ten first action classes"
CORPUS_REPO = "mteb/HMDB51"
CORPUS_REVISION = "50bb2abb741074ef0868e59ba51af710c825cd1f"  # refs/convert/parquet commit on the Hub
CORPUS_FILE = "default/test/0000.parquet"
CORPUS_BYTES = 480_591_821
CORPUS_ROWS = 1_530
CORPUS_ROW_GROUPS = 1  # of 5; 318 clips
CORPUS_LICENSE = "CC BY 4.0 (HMDB51, Serre Lab, Brown University; Kuehne, Jhuang, Garrote, Poggio, Serre, ICCV 2011; parquet repack mteb/HMDB51)"
CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"
# SHA-256 over the concatenated AVI bytes + UTF-8 label index of each row, in row order, and that byte total.
ROW_GROUP_PINS: dict[int, tuple[str, int]] = {
    0: ("c1eb3e9cd30abc8d3deadb1e82b982a57d2baf2035bfdf99d1343dcdb1c09bc1", 112_858_960),
}
DEFAULT_CACHE_DIR = Path("weights") / "hmdb51"

# The HMDB51 class vocabulary (parquet `label` feature names) and the ten classes the sample keeps, with the
# natural-language label the CLIP text tower is asked to rank (the upstream zero-shot recipe uses plain phrases).
HMDB51_CLASSES = ("brush_hair", "cartwheel", "catch", "chew", "clap", "climb", "climb_stairs", "dive", "draw_sword", "dribble", "drink", "eat", "fall_floor", "fencing", "flic_flac", "golf", "handstand", "hit", "hug", "jump", "kick", "kick_ball", "kiss", "laugh", "pick", "pour", "pullup", "punch", "push", "pushup", "ride_bike", "ride_horse", "run", "shake_hands", "shoot_ball", "shoot_bow", "shoot_gun", "sit", "situp", "smile", "smoke", "somersault", "stand", "swing_baseball", "sword", "sword_exercise", "talk", "throw", "turn", "walk", "wave")
SAMPLE_CLASS_TEXT: dict[str, str] = {
    "brush_hair": "brushing hair",
    "cartwheel": "doing a cartwheel",
    "catch": "catching a ball",
    "chew": "chewing",
    "clap": "clapping hands",
    "climb": "climbing",
    "climb_stairs": "climbing stairs",
    "dive": "diving into water",
    "draw_sword": "drawing a sword",
    "dribble": "dribbling a basketball",
}
SAMPLE_LABELS: tuple[str, ...] = tuple(SAMPLE_CLASS_TEXT.values())
SAMPLE_CLIPS_PER_CLASS = 30

SAMPLE_SEED = 42
SAMPLE_SPLIT = {"test": 6, "validation": 4}  # clips per class the source-grouped split reserves at least; the rest train
SAMPLE_DIGEST = "ddf8c1730d64ff80b5ec691522f09461220c6817dee70b55197cfd12b9191f4e"  # dataset_digest over the three default splits together; tests pin it
MIN_RECORDS = 16
MAX_RECORDS = 5_000
MIN_PER_CLASS = 2
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_CLIP_SUFFIXES = (".avi", ".mp4", ".mov", ".mkv", ".webm", ".gif", ".webp")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to read a
    parquet footer and one row group without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}", "User-Agent": "xclip-video-classification-pipeline"})
        with urllib.request.urlopen(request, timeout=600) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _declared_size(url: str) -> int:
    request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "xclip-video-classification-pipeline"})
    with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 (pinned https URL)
        length = response.headers.get("Content-Length")
    if length is None:
        raise ValueError(f"{url}: no Content-Length in the HEAD response")
    return int(length)


def _group_digest(rows: Sequence[Mapping[str, Any]]) -> tuple[str, int]:
    digest, total = hashlib.sha256(), 0
    for row in rows:
        data = row["video"]["bytes"]
        label = str(row["label"]).encode("utf-8")
        digest.update(data)
        digest.update(label)
        total += len(data) + len(label)
    return digest.hexdigest(), total


def fetch_corpus(
    *, cache_dir: str | Path | None = None, groups: Sequence[int] | None = None, opener: Any = None
) -> dict[int, list[dict[str, Any]]]:
    """Return the pinned row groups as lists of `{video, path, label}` (AVI bytes, file name, class index), from the
    cache (one parquet file per row group) or the Hub (footer + the row group, over range requests). Every row group's
    content is refused unless its SHA-256 and byte total match `ROW_GROUP_PINS`; a fresh fetch also checks the shard's
    declared size and row count."""
    import pyarrow.parquet as pq

    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    wanted = list(groups) if groups is not None else sorted(ROW_GROUP_PINS)
    out: dict[int, list[dict[str, Any]]] = {}
    reader = None
    for group in wanted:
        if group not in ROW_GROUP_PINS:
            raise ValueError(f"row group {group} has no pin; pinned groups are {sorted(ROW_GROUP_PINS)}")
        local = cache / f"test-rg{group}.parquet"
        rows: list[dict[str, Any]] | None = None
        if local.is_file():
            rows = pq.read_table(local).to_pylist()
            if _group_digest(rows) != ROW_GROUP_PINS[group]:
                rows = None  # stale or corrupt cache: refetch
        if rows is None:
            if reader is None:
                if opener is not None:
                    reader = pq.ParquetFile(opener(CORPUS_URL))
                else:
                    declared = _declared_size(CORPUS_URL)
                    if declared != CORPUS_BYTES:
                        raise ValueError(f"{CORPUS_FILE}: declared size {declared} != pinned {CORPUS_BYTES}")
                    reader = pq.ParquetFile(io.BufferedReader(_HttpRangeFile(CORPUS_URL, CORPUS_BYTES), buffer_size=1 << 20))
                if reader.metadata.num_rows != CORPUS_ROWS:
                    raise ValueError(f"{CORPUS_FILE}: {reader.metadata.num_rows} rows, pinned {CORPUS_ROWS}")
            table = reader.read_row_group(group, columns=["video", "label"])
            rows = table.to_pylist()
            digest, total = _group_digest(rows)
            if (digest, total) != ROW_GROUP_PINS[group]:
                raise ValueError(f"{CORPUS_FILE} row group {group}: sha256 {digest} / {total} bytes != pinned {ROW_GROUP_PINS[group]}")
            pq.write_table(table, local)
        out[group] = [{"video": r["video"]["bytes"], "path": str(r["video"]["path"] or ""), "label": int(r["label"])} for r in rows]
    return out


def decode_clip(data: bytes, n: int = NUM_FRAMES) -> list[Image.Image]:
    """Decode a video file held in memory with PyAV and return `n` uniformly spaced RGB frames (first and last
    included). Every frame is decoded (the container's declared frame count is not trusted); only the chosen ones
    are kept."""
    import av

    frames: list[Image.Image] = []
    with av.open(io.BytesIO(data)) as container:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        for frame in container.decode(stream):
            frames.append(frame.to_image())
    if len(frames) < n:
        raise ValueError(f"clip holds {len(frames)} frames, fewer than the {n} a clip needs")
    return sample_frames(frames, n)


def source_of(path: str, class_name: str) -> str:
    """The source an HMDB51 clip was cut from, keyed per action: `<source video>/<class>` from the file name
    `<source>_<class>_<tags>_<n>.avi`. Clips cut from one source video for one action share its scene and actor, so
    a split is made disjoint on this key (one film can still contribute different actions to different splits)."""
    match = re.match(rf"^(.*)_{re.escape(class_name)}_(.*)_(\d+)\.avi$", path)
    return f"{match.group(1) if match else Path(path).stem}/{class_name}"


def read_corpus(groups: Mapping[int, Sequence[Mapping[str, Any]]]) -> list[dict[str, Any]]:
    """Decode the verified row groups into records (one per clip of the ten sample classes; the partial eleventh
    class is dropped)."""
    out = []
    for group in sorted(groups):
        for index, row in enumerate(groups[group]):
            class_name = HMDB51_CLASSES[row["label"]]
            if class_name not in SAMPLE_CLASS_TEXT:
                continue
            out.append(
                {
                    "id": f"hmdb51-test-{group}-{index}",
                    "frames": decode_clip(row["video"]),
                    "label": SAMPLE_CLASS_TEXT[class_name],
                    "class": class_name,
                    "source": source_of(row["path"], class_name),
                    "source_row_group": group,
                }
            )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """Seeded source-grouped draw, per class: whole source videos go to the test split until it holds at least
    `sizes['test']` clips of that class, then to validation until `sizes['validation']`, and the rest train. Sources
    are taken smallest first (seeded order among equals), so the held-out splits stay near their targets and the
    largest sources train."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_class: dict[str, dict[str, list[dict[str, Any]]]] = defaultdict(lambda: defaultdict(list))
    for record in records:
        by_class[record["label"]][record.get("source") or record["id"]].append(dict(record))
    out: dict[str, list[dict[str, Any]]] = {"train": [], "validation": [], "test": []}
    for label in sorted(by_class):
        sources = sorted(by_class[label])
        rng.shuffle(sources)
        sources.sort(key=lambda source: len(by_class[label][source]))  # stable: seeded order among equal sizes
        counts = {"test": 0, "validation": 0, "train": 0}
        for source in sources:
            clips = by_class[label][source]
            if counts["test"] < sizes["test"]:
                out["test"].extend(clips)
                counts["test"] += len(clips)
            elif counts["validation"] < sizes["validation"]:
                out["validation"].extend(clips)
                counts["validation"] += len(clips)
            else:
                out["train"].extend(clips)
                counts["train"] += len(clips)
        if counts["train"] < MIN_PER_CLASS or counts["test"] < sizes["test"]:
            raise ValueError(f"class {label!r} has too few sources to reserve {sizes} clips and keep a training source")
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)


# ---------------------------------------------------------------------------------------------------------
# Record contract
# ---------------------------------------------------------------------------------------------------------


def _check_record(record: Any, index: int, labels: Sequence[str] | None) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/frames/label")
    for key in ("id", "frames", "label"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    try:
        frames = validate_clip(record["frames"])
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{where}: {exc}") from exc
    if not isinstance(record["label"], str) or not record["label"].strip():
        raise ValueError(f"{where}: label must be a non-empty str")
    label = format_labels([record["label"], "__other__"])[0]
    if labels is not None and label not in labels:
        raise ValueError(f"{where}: label {label!r} is not in the label set {list(labels)}")
    item = {"id": rid, "frames": frames, "label": label}
    for key in ("class", "source", "source_row_group"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    labels: Sequence[str] | None = None,
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    min_per_class: int = MIN_PER_CLASS,
) -> dict[str, Any]:
    """Structural validation of a labelled-clip dataset; raises ValueError before any model import. With `labels`
    every record's label must belong to that closed set; without, the set is the labels the records carry."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, frames, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    label_set = list(format_labels(labels)) if labels is not None else None
    checked, ids = [], set()
    for index, record in enumerate(records):
        item = _check_record(record, index, label_set)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    counts: dict[str, int] = defaultdict(int)
    for item in checked:
        counts[item["label"]] += 1
    if label_set is None:
        label_set = sorted(counts)
        if len(label_set) < 2:
            raise ValueError("a labelled-clip dataset needs at least two distinct labels")
    thin = [label for label, n in counts.items() if n < min_per_class]
    if thin:
        raise ValueError(f"every label needs at least {min_per_class} clips; too few for {sorted(thin)}")
    widths = [r["frames"][0].width for r in checked]
    heights = [r["frames"][0].height for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "labels": list(label_set),
        "per_label": {label: counts.get(label, 0) for label in label_set},
        "n_sources": len({r.get("source") or r["id"] for r in checked}),
        "frames_per_clip": NUM_FRAMES,
        "frame_width": {"min": min(widths), "max": max(widths)},
        "frame_height": {"min": min(heights), "max": max(heights)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def clip_digest(frames: Sequence[Image.Image]) -> str:
    """SHA-256 of the decoded RGB pixels of every frame (size-prefixed) — the identity a split is made disjoint on."""
    digest = hashlib.sha256()
    for frame in frames:
        rgb = frame.convert("RGB")
        digest.update(f"{rgb.width}x{rgb.height}:".encode())
        digest.update(rgb.tobytes())
    return digest.hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over (id, clip digest, label)."""
    parts = sorted(f"{r['id']}:{clip_digest(r['frames'])}:{r['label']}" for r in records)
    return _sha256_bytes("\n".join(parts).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no clip (by decoded-pixel digest) and no source video appears in two splits (leakage check)."""
    seen_clip: dict[str, str] = {}
    seen_source: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = clip_digest(record["frames"])
            if key in seen_clip and seen_clip[key] != name:
                raise ValueError(f"clip {record['id']!r} appears in both {seen_clip[key]} and {name}")
            seen_clip[key] = name
            source = record.get("source")
            if source:
                if source in seen_source and seen_source[source] != name:
                    raise ValueError(f"source video {source!r} has clips in both {seen_source[source]} and {name}")
                seen_source[source] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded, label-stratified shuffle of a BYOD dataset into train/validation/test after de-duplicating clips;
    records that carry a `source` keep every clip of one source in one split."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = clip_digest(record["frames"])
        if key not in seen:
            seen.add(key)
            unique.append(record)
    rng = random.Random(seed)
    out: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    by_label: dict[str, dict[str, list[dict[str, Any]]]] = defaultdict(lambda: defaultdict(list))
    for record in unique:
        by_label[record["label"]][record.get("source") or record["id"]].append(record)
    for label in sorted(by_label):
        groups = sorted(by_label[label])
        rng.shuffle(groups)
        n = sum(len(by_label[label][g]) for g in groups)
        n_test, n_val = max(1, round(n * test_fraction)), round(n * val_fraction)
        counts = {"test": 0, "validation": 0}
        for group in groups:
            clips = by_label[label][group]
            if counts["test"] < n_test:
                out["test"].extend(clips)
                counts["test"] += len(clips)
            elif counts["validation"] < n_val:
                out["validation"].extend(clips)
                counts["validation"] += len(clips)
            else:
                out["train"].extend(clips)
    if not out["train"]:
        raise ValueError(f"{len(unique)} distinct clips are too few to split into train/validation/test")
    return out


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a directory or zip holding video clips (any container PyAV or Pillow decodes: AVI, MP4, MOV,
    MKV, WebM, GIF, WebP) and a `labels.csv` with the columns `file` and `label` (and optionally `id` and `source`);
    every clip file must have a label row and every row a clip. Each clip is decoded into NUM_FRAMES uniform frames."""
    source = Path(path)
    members: dict[str, bytes] = {}
    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file():
                members[file.name] = file.read_bytes()
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                if not info.is_dir():
                    members[Path(info.filename).name] = archive.read(info)  # flattened; no extractall
    else:
        raise ValueError(f"{source} is neither a directory nor a zip file")
    if "labels.csv" not in members:
        raise ValueError("BYOD data must include labels.csv with the columns file and label")
    rows = list(csv.DictReader(io.StringIO(members["labels.csv"].decode("utf-8-sig"))))
    if not rows or "file" not in rows[0] or "label" not in rows[0]:
        raise ValueError("labels.csv must have the columns file and label")
    out = []
    for row in rows:
        name = Path(str(row.get("file", "")).strip()).name
        if name not in members:
            raise ValueError(f"labels.csv names a missing clip: {name}")
        try:
            if name.lower().endswith((".gif", ".webp")):
                pass  # standalone rewrite (build_notebook.py): `from .pipeline import frames_from_animation` removed — names are kernel globals defined by the carried modules

                frames = sample_frames(frames_from_animation(Image.open(io.BytesIO(members[name]))))
            else:
                frames = decode_clip(members[name])
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"BYOD file is not a decodable clip of at least {NUM_FRAMES} frames: {name}") from exc
        rid = str(row.get("id", "") or "").strip()
        item = {"id": rid or re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "frames": frames, "label": str(row.get("label", ""))}
        if str(row.get("source", "") or "").strip():
            item["source"] = str(row["source"]).strip()
        out.append(item)
    listed = {Path(str(r.get("file", "")).strip()).name for r in rows}
    unlisted = [n for n in members if n != "labels.csv" and n.lower().endswith(_CLIP_SUFFIXES) and n not in listed]
    if unlisted:
        raise ValueError(f"{len(unlisted)} clip file(s) have no labels.csv row, e.g. {unlisted[0]}")
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A summary table (id, frame size, label, class, source, provenance) in the BYOD `labels.csv` column layout plus
    extras (`file` names the id; the frames themselves are not written)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "file", "width", "height", "frames", "label", "class", "source", "source_row_group"])
        for r in records:
            writer.writerow([r["id"], f"{r['id']}.avi", r["frames"][0].width, r["frames"][0].height, len(r["frames"]), r["label"], r.get("class", ""), r.get("source", ""), r.get("source_row_group", "")])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `9`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a2e27a78a2b5…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `XClipVideoClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "xclip-base-patch32",
  "modelId": "microsoft/xclip-base-patch32",
  "revision": "a2e27a78a2b5d802e894b8a1ef14f3a8ce490963",
  "files": [
    {
      "path": "README.md",
      "bytes": 2728,
      "sha256": "5b4c29da30c14c41dadc2320cedb68d7bc30147aa171c04ce8d3d7b8acf8da91"
    },
    {
      "path": "config.json",
      "bytes": 4718,
      "sha256": "13bb919d1ef16f3b80b03cdf5c575d688dfafd21d12566849cc557b00e058ce3"
    },
    {
      "path": "merges.txt",
      "bytes": 524657,
      "sha256": "f526393189112391ce6f9795d4695f704121ce452c3aad1f5335cc41337eba85"
    },
    {
      "path": "model.safetensors",
      "bytes": 786414772,
      "sha256": "abf286e8cdd0612761c3e42d3a55eca998382dfa67a04a0f3fdcdfa4f150cdbb"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 309,
      "sha256": "c14b2b5c8f26a754df62235ba79d1ca63cfdd9b3de76ee688e4a30ea1e5c6986"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 472,
      "sha256": "c4864a9376a8401918425bed71fc14fc0e81f9b59ec45c1cf96cccb2df508eac"
    },
    {
      "path": "tokenizer.json",
      "bytes": 2224041,
      "sha256": "a75dc79c6ec004a7e2d346c20e0af8d29aa2b251ea356964718aef8b8f052e80"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 965,
      "sha256": "7e28810d55aafb02f6b216f05fa1208a4aac62abf794796288a70314eed5ddf3"
    },
    {
      "path": "vocab.json",
      "bytes": 862328,
      "sha256": "5047b556ce86ccaf6aa22b3ffccfc52d391ea4accdab9c2f2407da5b742d4363"
    }
  ],
  "totalBytes": 790034990
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = XClipVideoClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. HMDB51 clips, the labels and the source-grouped split

`fetch_corpus` returns the pinned row group from the cache under `weights/hmdb51/` or the Hub at the pinned parquet-conversion revision — `pyarrow` reads the shard's footer and exactly that row group over HTTPS range requests; the cached file is re-hashed and a fetched row group refused on any SHA-256 or byte-total mismatch — and `read_corpus` turns each row of the ten sample classes into a record: the AVI decoded by PyAV into 8 uniformly spaced frames (mostly 320 × 240), the class rendered as the natural-language label the text tower is asked to rank, and the source video the clip was cut from. `build_sample_dataset` draws a seeded **source-grouped** split: per class, whole source videos go to the test split until it holds six clips, then to validation until four, and the rest train (181 / 53 / 66 in the build record). `validate_dataset` then checks every record against the contract and the label set, `check_split_disjoint` asserts no clip (by decoded-pixel digest) and no source video is shared, and the training split's summary table is written to `outputs/xclip_video_classification_train.csv`.

Look for: 300 clips of ten labels, three digests, and four refusal probes — a duplicate id, a label outside the set, a clip with three frames, and a dataset too small to use — each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    LABELS = validate_dataset(records)['labels']
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    t0 = time.perf_counter()
    corpus_groups = fetch_corpus(cache_dir='weights/hmdb51')
    fetch_seconds = round(time.perf_counter() - t0, 1)
    corpus = read_corpus(corpus_groups)
    LABELS = list(SAMPLE_LABELS)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} @ {CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'row_groups': len(corpus_groups), 'rows': sum(len(v) for v in corpus_groups.values()), 'clips_kept': len(corpus), 'bytes': sum(len(r['video']) for v in corpus_groups.values() for r in v), 'fetch_seconds': fetch_seconds, 'decode_seconds': round(time.perf_counter() - t0 - fetch_seconds, 1)}
dataset_manifests = {name: validate_dataset(part, LABELS, min_records=1, min_per_class=1) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
validate_dataset(train_records, LABELS)  # the training split must satisfy the full record bounds
write_dataset_csv(train_records, 'outputs/xclip_video_classification_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'labels': LABELS})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'per_label': manifest['per_label'], 'sources': manifest['n_sources'], 'width': manifest['frame_width'], 'height': manifest['frame_height'], 'digest': manifest['digest'][:16] + '...'}})


def frame_strip(frames, height=120):
    tiles = [f.resize((round(f.width * height / f.height), height)) for f in frames]
    strip = Image.new('RGB', (sum(t.width for t in tiles) + 4 * (len(tiles) - 1), height), (255, 255, 255))
    x = 0
    for tile in tiles:
        strip.paste(tile, (x, 0))
        x += tile.width + 4
    return strip


example = train_records[0]
frame_strip(example['frames']).save('outputs/xclip_video_classification_example_clip.png')
print({'example': {'id': example['id'], 'frames': len(example['frames']), 'frame_size': list(example['frames'][0].size), 'label': example['label'], 'source': example.get('source')}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:16]],
    'label outside the set': [{**train_records[0], 'label': 'juggling'}, *train_records[1:16]],
    'clip with three frames': [{**train_records[0], 'frames': train_records[0]['frames'][:3]}, *train_records[1:16]],
    'too small': train_records[:8],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe, LABELS)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Rank five drawn clips through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: five deterministic 8-frame clips at 320 × 240 drawn with Pillow — a ball rolling right, a ball bouncing, a square growing, a sun setting, a ball standing still — with the five class names that describe them; a different clip family from the human-action videos, and clips the adapted model will be asked to rank again in Section 9. `validate_inputs` applies exactly the checks `classify` applies (exactly `NUM_FRAMES` frames of one size within the side ceilings, 2..`MAX_LABELS` distinct names of at most `MAX_LABEL_CHARS` characters) and one combined manifest records the request; a three-frame clip is validated too and its rejection recorded as a finding. `classify` returns one probability per supplied name — **the probabilities are a softmax over your label set**, and the label set is a **caller-owned request parameter**. The per-request `evaluation_report` with the intended classes is `sample-sanity`: `top1_accuracy` against the chance baseline on five cartoons is plumbing evidence, not a video-classification benchmark — video-classification accuracy needs labelled clips of the deployment domain, which Section 6 supplies. The inference-only card recorded 1/5 (chance 1/5): the model was trained on human-action video and does not read the motion of flat drawn shapes, which the notebook keeps as a finding rather than tuning the drawings until they pass.

In [ ]:
def synthetic_clip(kind, n=8, size=(320, 240)):
    """An 8-frame cartoon clip drawn with Pillow (no text): sky, green ground and one moving element."""
    frames = []
    for index in range(n):
        t = index / (n - 1)
        sky = (135, 206, 235)
        if kind == 'the sun setting':
            sky = (int(135 * (1 - t) + 30 * t), int(206 * (1 - t) + 40 * t), int(235 * (1 - t) + 80 * t))
        frame = Image.new('RGB', size, sky)
        d = ImageDraw.Draw(frame)
        d.rectangle([0, 170, 320, 240], fill=(60, 179, 75))  # ground
        if kind == 'a ball rolling to the right':
            x = 30 + t * 230
            d.ellipse([x, 130, x + 40, 170], fill=(220, 40, 40))
        elif kind == 'a ball bouncing up and down':
            y = 130 - abs(np.sin(t * np.pi * 2)) * 100
            d.ellipse([140, y, 180, y + 40], fill=(220, 40, 40))
        elif kind == 'a square growing larger':
            s = 10 + t * 90
            d.rectangle([160 - s / 2, 120 - s / 2, 160 + s / 2, 120 + s / 2], fill=(40, 70, 200))
        elif kind == 'the sun setting':
            y = 30 + t * 140
            d.ellipse([240, y, 290, y + 50], fill=(255, 215, 0))
        elif kind == 'a ball standing still':
            d.ellipse([140, 130, 180, 170], fill=(220, 40, 40))
        frames.append(frame)
    return frames


DRAWN_LABELS = ['a ball rolling to the right', 'a ball bouncing up and down', 'a square growing larger', 'the sun setting', 'a ball standing still']
drawn_clips = [synthetic_clip(kind) for kind in DRAWN_LABELS]
drawn_names = [f"synthetic_{kind.replace(' ', '_')}_8x320x240" for kind in DRAWN_LABELS]
drawn_digests = {name: hashlib.sha256(b''.join(np.asarray(frame.convert('RGB')).tobytes() for frame in clip)).hexdigest() for name, clip in zip(drawn_names, drawn_clips, strict=True)}
print({'ceilings': {'NUM_FRAMES': NUM_FRAMES, 'FRAME_SIZE': FRAME_SIZE, 'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MIN_LABELS': MIN_LABELS, 'MAX_LABELS': MAX_LABELS, 'MAX_LABEL_CHARS': MAX_LABEL_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'MIN_PER_CLASS': MIN_PER_CLASS, 'EVAL_BATCH_SIZE': EVAL_BATCH_SIZE, 'device': pipe.device}})
input_manifest = validate_inputs(drawn_clips, DRAWN_LABELS, names=drawn_names)
try:
    validate_inputs([drawn_clips[0][:3]], DRAWN_LABELS)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'three-frame-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/xclip_video_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'drawn_clips': len(drawn_clips), 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def rank_drawings(pipeline, label):
    results, timings = [], []
    for name, clip in zip(drawn_names, drawn_clips, strict=True):
        started = time.perf_counter()
        result = pipeline.classify(clip, DRAWN_LABELS)
        timings.append(round(time.perf_counter() - started, 3))
        results.append({**result, 'clip': name, 'rgb_sha256': drawn_digests[name]})
    checks = {
        'probabilities_sum_to_one': all(abs(sum(p['probability'] for p in r['predictions']) - 1.0) < 1e-6 for r in results),
        'one_entry_per_label': all([p['label'] for p in r['predictions']] and sorted(p['label'] for p in r['predictions']) == sorted(r['labels']) for r in results),
        'ranked_descending': all(all(a['probability'] >= b['probability'] for a, b in zip(r['predictions'], r['predictions'][1:], strict=False)) for r in results),
        'identity_reported': all(r['model_id'] == MODEL_ID and r['model_revision'] == MODEL_REVISION for r in results),
    }
    if not all(checks.values()):
        raise RuntimeError(f'classify output failed a sanity check: {checks}')
    report = evaluation_report(results, DRAWN_LABELS, sample_kind='synthetic (drawn in this notebook)')
    with open(f'outputs/xclip_video_classification_drawing_{label}.json', 'w', encoding='utf-8') as handle:
        json.dump({'results': results, 'report': report}, handle, indent=2, ensure_ascii=False)
    print({label: {'seconds': timings, 'checks': checks, 'top1': [(r['clip'].split('_8x')[0], r['top1'], round(r['predictions'][0]['probability'], 3)) for r in results], 'verdict': report['verdict'], 'top1_accuracy': report['metrics'][0]['value'], 'chance': report['baselines'][0]['value']}})
    return results, timings, checks, report


frozen_drawn, frozen_drawn_timings, frozen_drawn_checks, frozen_drawn_report = rank_drawings(pipe, 'frozen')

## 6. Baselines and the frozen model on the test clips

Two non-adapted baselines frame the adaptation, each scored by `classification_metrics` (carried in `metrics.py`): **top-1 accuracy** (the highest-scoring name is the reference label), **top-3 accuracy** (the reference is among the three highest), **macro recall** (the mean per-label recall, so a class the model never predicts counts fully) and **macro F1**, with the per-label confusion beside them. The **chance** baseline guesses uniformly: top-1 = 1 / 10 by construction. The **majority-label** baseline predicts the most frequent training label for every clip: what the label distribution buys without looking at the frames. The **frozen model** is scored by `pipe.evaluate`, which ranks the closed label set for every clip in batches of `EVAL_BATCH_SIZE` and returns the rankings with the rates. Expect the frozen model **well above both baselines** — it is a zero-shot action classifier and these are human actions: the build record measured **@P:FROZEN_TOP1@** top-1 on the 66 held-out clips, with the per-label recall showing which classes it misses (@P:FROZEN_WEAK@); read four rankings under their references.

In [ ]:
METRICS = ('top1_accuracy', 'top3_accuracy', 'macro_recall', 'macro_f1')

baseline_chance = chance_baseline(test_records, LABELS)
baseline_majority = majority_baseline(train_records, test_records, LABELS)
print({'chance_baseline': {k: round(baseline_chance[k], 3) for k in METRICS}, 'n': baseline_chance['n'], 'n_labels': baseline_chance['n_labels']})
print({'majority_baseline': {k: round(baseline_majority[k], 3) for k in METRICS}, 'label': baseline_majority['label']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, LABELS, batch_size=EVAL_BATCH_SIZE)
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'n': frozen_test['n'], 'mean_rank': round(frozen_test['mean_rank'], 2), 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'per_label_recall': {label: round(row['recall'], 2) for label, row in frozen_test['per_label'].items()}})
print({'definitions': frozen_test['definitions']})
for record, ranking, probability in zip(test_records[:4], frozen_test['rankings'][:4], frozen_test['top1_probability'][:4], strict=True):
    print({'id': record['id'], 'reference': record['label'], 'frozen_top3': ranking[:3], 'top1_probability': round(probability, 3)})

## 7. Bounded fine-tuning of the fusion head

`pipe.adapt` trains only the fusion head — the multi-frame integration transformer, the visual projection, the prompt-side visual layer norm and projection, and the video-specific prompt generator: 10,247,680 of 196,585,729 parameters — while the ViT-B/32 vision tower with its cross-frame attention, the CLIP text tower, the text projection and the logit scale stay frozen. The loss is the **cross-entropy over the closed label set**: the model's own contrastive scores, read as a classifier over the ten names. Because both towers are frozen, their outputs — the pooled CLS and the 49 patch features of every frame, and the ten label embeddings — are computed once under no gradient and cached (the **frozen-tower cache**), and each step runs only the head on those cached features: the logits equal the full model's exactly, at a fraction of the cost. AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, seeded shuffling, no scheduler, no augmentation. Epoch 0 records the frozen model's validation rates; every epoch is scored on the 53 validation clips, and the epoch with the **highest validation top-1 accuracy** (the earliest on ties) is kept.

Watch the validation top-1 rise from @P:VAL_TOP1_0@ to @P:VAL_TOP1_BEST@ (epoch @P:BEST_EPOCH@ in the build record) while the loss drops from about @P:LOSS_1@ to @P:LOSS_LAST@: @P:ADAPTED_READ@. The learning rate is deliberately small — a head this size memorises 181 clips within an epoch or two at 1e-4, and the validation curve is then flat from the first epoch.

In [ ]:
EPOCHS = 8  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in METRICS})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, LABELS, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'labels': adapt_result['labels'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'objective': adapt_result['objective'], 'cache_seconds': adapt_result['cache_seconds'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test clips were never used for training or epoch selection, and no clip and no source video appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6 and the four systems are put side by side. Read it in this order: **top-1 accuracy** first (the measure the epoch was selected on — the build record measured @P:FROZEN_TOP1@ → **@P:ADAPTED_TOP1@**), then **macro F1** (@P:FROZEN_F1@ → @P:ADAPTED_F1@), then **top-3 accuracy** (@P:FROZEN_TOP3@ → @P:ADAPTED_TOP3@), then the per-label recall to see which classes moved (@P:ADAPTED_WEAK@). The cell asserts the adapted top-1 is at least the frozen one and above chance. Sixty-six clips from one seeded split give **no dispersion estimate** — one clip is 1.5 points — so the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a result on ten HMDB51 actions says nothing about other actions, other cameras or your videos until you measure them.

In [ ]:
adapted_test = pipe.evaluate(test_records, LABELS, batch_size=EVAL_BATCH_SIZE)
adapted_val = pipe.evaluate(val_records, LABELS, batch_size=EVAL_BATCH_SIZE)
comparison = {metric: {'chance': round(baseline_chance[metric], 3), 'majority': round(baseline_majority[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
comparison['per_label_recall'] = {label: {'frozen': round(frozen_test['per_label'][label]['recall'], 2), 'adapted': round(adapted_test['per_label'][label]['recall'], 2), 'support': adapted_test['per_label'][label]['support']} for label in LABELS}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'labels': LABELS,
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'chance': baseline_chance, 'majority': {k: v for k, v in baseline_majority.items() if k not in ('definitions',)}},
    'frozen_test': {k: v for k, v in frozen_test.items() if k != 'definitions'},
    'validation_metrics': {k: v for k, v in adapted_val.items() if k != 'definitions'},
    'test_metrics': adapted_test,
    'per_clip': [{'id': r['id'], 'reference': r['label'], 'source': r.get('source'), 'frozen_top1': f, 'frozen_ranking': fr, 'adapted_top1': a, 'adapted_ranking': ar} for r, f, fr, a, ar in zip(test_records, frozen_test['predictions'], frozen_test['rankings'], adapted_test['predictions'], adapted_test['rankings'], strict=True)],
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/xclip_video_classification_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['top1_accuracy'] >= frozen_test['top1_accuracy']
assert adapted_test['top1_accuracy'] > baseline_chance['top1_accuracy']
print({'report': 'outputs/xclip_video_classification_evaluation_report.json', 'adapted_beats_both_baselines': adapted_test['top1_accuracy'] > max(baseline_chance['top1_accuracy'], baseline_majority['top1_accuracy'])})

## 9. Look at the clips, rank the drawings again, export the adapter and reload it

Six held-out clips are written as panels (`outputs/xclip_video_classification_examples/`: the 8-frame strip with the reference label, the frozen top-3 and the adapted top-3 beneath it) so the numbers can be checked by eye. The five drawn clips from Section 5 are then ranked again by the adapted model — the fusion head that was tuned ranks every request, so this is a small look at what the adaptation did *outside* its label set and its corpus: the build record measured @P:DRAWING_AFTER@ — five cartoons of evidence, not a measurement.

`pipe.save_artifact` writes the trained tensors — the fusion head, about 41 MB in float32 — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the label set the head was trained on, the training configuration and the epoch history (OUT8). `XClipVideoClassificationPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the fusion head, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical rankings on eight test clips (VER4).

In [ ]:
import shutil

examples_dir = Path('outputs/xclip_video_classification_examples')
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir(parents=True)
caption_font = ImageFont.load_default(size=18)
for record, frozen_ranking, adapted_ranking in zip(test_records[:6], frozen_test['rankings'][:6], adapted_test['rankings'][:6], strict=True):
    strip = frame_strip(record['frames'], height=120)
    sheet = Image.new('RGB', (max(strip.width, 1400), strip.height + 96), (255, 255, 255))
    sheet.paste(strip, (0, 0))
    marker = ImageDraw.Draw(sheet)
    for i, (tag, text) in enumerate((('REF', record['label']), ('FROZEN', ' > '.join(frozen_ranking[:3])), ('ADAPTED', ' > '.join(adapted_ranking[:3])))):
        marker.text((8, strip.height + 6 + i * 28), f'{tag}: {text[:140]}', fill=(20, 20, 20) if tag != 'FROZEN' else (150, 40, 40), font=caption_font)
    sheet.save(examples_dir / f"{record['id']}.png")
print({'examples': sorted(p.name for p in examples_dir.iterdir()), 'rows': ['reference', 'frozen top-3', 'adapted top-3']})

adapted_drawn, adapted_drawn_timings, adapted_drawn_checks, adapted_drawn_report = rank_drawings(pipe, 'adapted')

artifact_dir = Path('outputs/xclip_video_classification_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'xclip_video_classification', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'labels': artifact_manifest['adapter']['labels'], 'best_epoch': artifact_manifest['adapter']['best_epoch']})

reloaded = XClipVideoClassificationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [[p['label'] for p in item['predictions']] for item in pipe.classify_batch([r['frames'] for r in test_records[:8]], LABELS)]
after = [[p['label'] for p in item['predictions']] for item in reloaded.classify_batch([r['frames'] for r in test_records[:8]], LABELS)]
parity = {'identical_rankings': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_rankings'] == parity['of']

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'labels': LABELS,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'file': CORPUS_FILE, 'license': CORPUS_LICENSE, 'row_groups': sorted(ROW_GROUP_PINS), 'shard_bytes': CORPUS_BYTES, 'classes': list(SAMPLE_CLASS_TEXT)},
    'inference_contract': {'input_manifest': input_manifest, 'drawn': {'names': drawn_names, 'labels': DRAWN_LABELS, 'digests': drawn_digests}, 'frozen': {'results': frozen_drawn, 'seconds': frozen_drawn_timings, 'checks': frozen_drawn_checks, 'report': frozen_drawn_report}, 'adapted': {'results': adapted_drawn, 'seconds': adapted_drawn_timings, 'checks': adapted_drawn_checks, 'report': adapted_drawn_report}, 'output_files': ['outputs/xclip_video_classification_drawing_frozen.json', 'outputs/xclip_video_classification_drawing_adapted.json']},
    'comparison': comparison,
    'examples': 'outputs/xclip_video_classification_examples',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'av': av.__version__, 'device': pipe.device, 'dtype': 'float32'},
}
with open('outputs/xclip_video_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A zero-shot video–text model trained on Kinetics-400 already ranks the right one of ten HMDB51 actions first for @P:FROZEN_TOP1@ of the held-out clips; a bounded fine-tuning of its fusion head on 181 clips moves that to @P:ADAPTED_TOP1@ top-1 and @P:ADAPTED_F1@ macro F1 in the build record, with a 41 MB adapter that reloads ranking-for-ranking. That is the claim: the adaptation contract works end to end on a closed label set with a real labelled set, and the numbers it produces are read as top-1 / top-3 accuracy, macro recall and macro F1 against two non-adapted baselines and the frozen model, with the per-label recall beside them rather than in isolation. @P:SIBLING_COMPARISON@

The test split is 66 clips from one seeded, source-grouped draw of one 300-clip sample, the validation split that picks the epoch is 53, and every rate is over one reference label per clip — not a benchmark, not the HMDB51 protocol (three splits over all 51 classes), not a measure of temporal localisation. So a result here says the contract works on ten actions cut from films, not that the adapted model handles other actions, other cameras or your clips. The head that was tuned ranks every request: the drawn clips re-ranked in Section 9 are five cartoons of evidence about what the tuning did outside its label set (@P:DRAWING_AFTER@), not a measurement, and a deployment that ranks other label sets must measure them after adapting. The towers were not adapted: what the frame encoder cannot see stays unseen, and **the probabilities remain a softmax over your label set**.

Three things to carry to real data. **Baselines first:** the chance and majority rates on *your* labels, and the frozen model's per-label recall, are the numbers to read before any adapted one. **Macro over micro:** a class the model never predicts costs a full tenth of macro recall while barely moving top-1 on a skewed set; read both. **Leakage:** keep every clip in one split (the contract de-duplicates by decoded pixels) and split by source video, camera or session when your clips come from few recordings — clips cut from one video share its scene and actor.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled clip set, validate the demonstrated dataset contract without leakage, execute the inference contract for five clips and a bounded fine-tuning of the fusion head with the closed-set cross-entropy, evaluate against two non-adapted baselines and the frozen model on a source-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, classification quality on any other action set or video domain, ranking quality on other label sets after adaptation, or production fitness.

**Optional experiments (they do not affect the default path):** raise `EPOCHS` and watch the validation top-1 pick the epoch; set `LEARNING_RATE` to `1e-4` and read a curve that peaks in the first epoch and then flattens as the head memorises the training clips; change `SPLIT_SEED` and read how much 66 clips move; edit `SAMPLE_CLASS_TEXT`'s phrasing in the carried module and rerun from Section 4 to see how much the frozen model depends on the wording of the label; or bring your own labelled clips through BYOD and read the two baselines before the adapted number.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/xclip-base-patch32/` and rerun Section 3. A `sha256` `ValueError` naming the parquet row group in Section 4: the cached `weights/hmdb51/test-rg0.parquet` is incomplete — delete it and rerun Section 4.

## References

- Repository README: https://github.com/kurtvalcorza/xclip-video-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/xclip-video-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/xclip-video-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/microsoft/xclip-base-patch32
- Upstream code: https://github.com/microsoft/VideoX/tree/master/X-CLIP
- Expanding Language-Image Pretrained Models for General Video Recognition (Ni et al., ECCV 2022): https://arxiv.org/abs/2208.02816
- HMDB51 (Serre Lab, CC BY 4.0): https://huggingface.co/datasets/Serrelab/hmdb51 — Kuehne, Jhuang, Garrote, Poggio, Serre, HMDB: A Large Video Database for Human Motion Recognition (ICCV 2011); parquet repack read here: https://huggingface.co/datasets/mteb/HMDB51
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)